# Phase 4 — Candidate-Conditioned Pair Matcher

This phase trains a first learned pair scorer on candidates from the Phase 3 blocking union. It uses blocking-route evidence flags, target-source identity, and per-query candidate-set size as pair features. Candidate positives and hard negatives come only from a deterministic sample of the non-validation Source 1 rows. A separate train-only calibration slice selects the probability threshold by macro F0.5. The fixed Phase 2 validation set is scored only after fitting and threshold selection.

The model is deliberately a lightweight baseline: it uses blocking-route evidence flags, target source, and candidate-set size. It does not yet use edit distance, TF-IDF, character-level string similarity, or external data. Its candidate recall ceiling is the Phase 3 union-token-pair recall.

## Results

The model trained on **498,357 candidate pairs** from **15,941 fit queries** (50,815 positives and 447,542 sampled hard negatives). A disjoint set of **4,059 training queries** selected the probability threshold **0.19129** by macro F0.5.

On the untouched 441,363-row validation holdout, macro F0.5 is **0.46887**, compared with **0.05585** for predicting no matches. The generated candidate pool has **92.14% target recall**; after scoring, micro precision is **22.40%** and micro recall is **47.03%**. Source-specific macro F0.5 is **0.48295 for S2** and **0.42311 for S3**. The gap and the relatively low micro precision show that route evidence alone is not enough for a final matcher.

The score is the README-defined macro F0.5 over all Source 1 entities, including singletons. Validation labels were used only for final evaluation; they were not used to fit weights or choose the threshold.

## Run

The full run builds disk-backed indexes for the 10.3 million S2/S3 training targets, trains on 20,000 sampled training queries, calibrates on 20% of those queries, and evaluates all 441,363 validation queries. It may take several minutes. Detailed results are written to `dataset/splits/phase4_matcher_metrics.json`. The next matcher iteration should add name/address similarity features and evaluate S2 and S3 separately, with threshold calibration kept inside the training partition.


In [ ]:
from pathlib import Path

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'notebooks' / 'validation' / 'phase4_pairwise_matcher.py').is_file())
script_path = project_root / 'notebooks' / 'validation' / 'phase4_pairwise_matcher.py'
exec(compile(script_path.read_text(encoding='utf-8'), str(script_path), 'exec'), {'__name__': '__main__'})


In [ ]:
import json
from pathlib import Path

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'dataset' / 'splits' / 'phase4_matcher_metrics.json').is_file())
results = json.loads((project_root / 'dataset' / 'splits' / 'phase4_matcher_metrics.json').read_text(encoding='utf-8'))
print(json.dumps(results, indent=2))
